## xLH-ai-bp-vs-hf
Eine exemplarische Umsetzung für die Integration von LLM-Modellen für die Berufspädagogik

In [18]:
import os
import pathlib
import sys
__cwd__ = str(pathlib.Path(os.getcwd())).replace('\\\\', '/')
sys.path.append(__cwd__)
print(__cwd__)


D:\Python\xLH-mims\notebooks\src\ai_bp_hf

In [33]:
import pandas as pd
from dataclasses import dataclass
from pydantic_ai import Tool
from pathlib import Path
from rich import print
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from tools import read_hk
from llm_model import get_model, LlmModel
from object_to_file import base_model_to_file
from config import config
import nest_asyncio
nest_asyncio.apply()

In [20]:
# OpenAi API Key (Credentials hinterlegt in der Datei ..\xLH-mims-data\config\xlh_mims_python.env)
print(config.openai_api_key[:20])
# falls keine Datei im lokalen Speicher vorhanden ist, wir die Datei env.py

sk-proj-hevDtsISJBCJ

In [21]:
# df = pd.read_pickle("RS_LK.pkl")
# df.head()

In [22]:
@dataclass
class Deps:
    name: str

In [45]:
class Punkte(BaseModel):
    titel: str = Field(description='Kurzbeschreibung')
    beschreibung: str = Field(description='ausführliche Beschreibung')
    begruendung: str = Field(description='Begründung')

In [46]:
class Quantity(BaseModel):
    HK_TS: int = Field(description='Gesamtanzahl der eindeutigen Handlungskompetenzen des Transportsanitäters')
    LK_TS: int = Field(description='Gesamtanzahl der eindeutigen Leistungskriterien des Transportsanitäters')
    HK_RS: int = Field(description='Gesamtanzahl der eindeutigen Handlungskompetenzen des Rettungssanitäters')
    LK_RS: int = Field(description='Gesamtanzahl der eindeutigen Leistungskriterien des Rettungssanitäters')
    text_gemeinsamkeiten: list[Punkte] = Field(description='as sind die Gemeinsamkeiten')
    text_unterschiede: list[Punkte] = Field(description='Was sind die Unterschiede')

In [47]:
model = get_model(llm_model=LlmModel.OPENAI_GPT_5_2)  # ToDo: Integration OLLAMA...
agent= Agent(
    model=model,
    system_prompt=('Du analysierst Daten zu den Aus- und Weiterbildungen der schweizer Berufsbildung.'),
    deps_type=Deps,
    tools=[
        # Tool(read_hk, takes_ctx=True),
    ],
    retries=3,
    output_type=Quantity,
)
deps = Deps(name='-')

In [48]:
ts_hk = Path("TS_HK.md").read_text(encoding="utf-8")
ts_lk = Path("TS_LK.md").read_text(encoding="utf-8")
rs_hk = Path("RS_HK.md").read_text(encoding="utf-8")
rs_lk = Path("RS_LK.md").read_text(encoding="utf-8")

response = agent.run_sync(user_prompt=f"Wie unterscheiden sich die beiden Weiterbildungen Rettungssanitäter (RS) und Transportsanitäter (TS)? "
                                      f"Hier sind die entsprechenden Daten dazu:"
                                      f"- hier die Handlungskompetenzen des RS: {rs_hk}"
                                      f"- hier die Leistungskriterien des RS in Verbindung zu den Handlungskompetenzen und Zuteilung des Niveaus: {rs_lk}"
                                      f"- hier die Handlungskompetenzen des TS: {ts_hk}"
                                      f"- hier die Leistungskriterien des TS in Verbindung zu den Handlungskompetenzen und Zuteilung des Niveaus: {ts_lk}"
                          )

In [49]:
result: Quantity = response.output
base_model_to_file(result)
# siehe recipe.json
print(f'Antwort: {result.model_dump_json(indent=4)[:]}')

Antwort: {
    "HK_TS": 22,
    "LK_TS": 168,
    "HK_RS": 20,
    "LK_RS": 242,
    "text_gemeinsamkeiten": [
        {
            "titel": "Einsatzplanung, Situationsüberblick und Risikomanagement",
            "beschreibung": "Beide Profile verlangen, Einsätze strukturiert vorzubereiten, vor Ort einen 
Situationsüberblick zu gewinnen, Gefahren zu erkennen und Schutz-/Präventionsmassnahmen umzusetzen. Dazu gehören 
das Bilden und laufende Überprüfen eines mentalen Modells, das Abwägen von Standards/Richtlinien sowie das Anpassen
des Handelns bei Lageänderungen.",
            "begruendung": "RS: rs_hk_01_01/01_02/01_03 (Einsatzmeldung beurteilen, Situation erfassen, mit Risiken
umgehen). TS: ts_hk_01_01/02_01/01_02 (Einsatz planen, Situationsüberblick, Schutz- und Präventionsmassnahmen)."
        },
        {
            "titel": "Kommunikation, Zusammenarbeit und strukturierte Übergaben",
            "beschreibung": "Beide Weiterbildungen betonen effiziente Teamkommunikation, Kooperation mit anderen 
Fachpersonen/Organisationen sowie strukturierte Übernahme- und Übergabeprozesse (Rapport), um Informationsverluste 
und Behandlungsfehler zu vermeiden.",
            "begruendung": "RS: rs_hk_02_01/02_02/02_03. TS: ts_hk_01_04 und ts_hk_03_04."
        },
        {
            "titel": "Patientenbeurteilung, Überwachung und (Basis-)Sofortmassnahmen",
            "beschreibung": "Beide Rollen müssen Patientinnen/Patienten systematisch beurteilen, überwachen und bei
kritischen Veränderungen reagieren. Lebensrettende Sofortmassnahmen sind in beiden Profilen verankert; beim TS mit 
Fokus auf Einleiten und Aufrechterhalten bis zur Übergabe, beim RS umfassender als Teil der eigenverantwortlichen 
präklinischen Behandlung.",
            "begruendung": "RS: rs_hk_03_01 bis 03_05. TS: ts_hk_02_02/02_06/02_03."
        },
        {
            "titel": "Rettung/Transfer, Lagerung/Transport und sicheres Führen von Einsatzfahrzeugen",
            "beschreibung": "Beide Profile enthalten Kompetenzen zu Rettungs-/Transfertechniken, patientengerechter
Lagerung und sicherem, angepasstem Führen von Einsatzfahrzeugen inkl. Positionierung und Berücksichtigung von 
Verkehr/Witterung/Patientenzustand.",
            "begruendung": "RS: rs_hk_04_01 bis 04_03. TS: ts_hk_03_01 bis 03_03."
        },
        {
            "titel": "Dokumentation, Material-/Fahrzeugbewirtschaftung und digitale Kompetenzen",
            "beschreibung": "Beide verlangen vollständige, objektive, datenschutzkonforme Dokumentation, 
Sicherstellung der Einsatzbereitschaft von Material/Fahrzeugen sowie kompetente Nutzung digitaler Hilfsmittel unter
Einhaltung des Datenschutzes.",
            "begruendung": "RS: rs_hk_01_05, 05_01, 05_02. TS: ts_hk_01_05, 04_01, 04_02, 04_03."
        },
        {
            "titel": "Selbstmanagement, Ethik/Recht, Qualität und Weiterentwicklung",
            "beschreibung": "Beide Profile adressieren Erhalt der eigenen Gesundheit/Belastungsbewältigung, 
ethisch-rechtlich korrektes Handeln, Mitwirkung an Qualitätssicherung sowie kontinuierliche berufliche 
Entwicklung/Fortbildung.",
            "begruendung": "RS: rs_hk_06_01 bis 06_05. TS: ts_hk_05_01 bis 05_04."
        }
    ],
    "text_unterschiede": [
        {
            "titel": "Patientenkollektiv und Verantwortungsgrad: RS eigenverantwortlich auch bei 
instabilen/kritischen Notfällen, TS primär stabile Transporte",
            "beschreibung": "Der zentrale Unterschied ist der Einsatzfokus: TS leiten und koordinieren planbare 
Einsätze mit stabilen Patientinnen/Patienten und übernehmen bei instabilen Patientinnen/Patienten eine 
Assistenzfunktion. RS beurteilen, priorisieren und behandeln Patientinnen/Patienten im präklinischen Umfeld 
eigenverantwortlich, inkl. kritischer Zustände, und leiten daraus Strategie (z.B. Zielort/Transportmittel) ab.",
            "begruendung": "TS: ts_hk_01_03 (Leiten stabiler Einsätze), ts_hk_02_05 (Assistenz bei instabilen). RS:
rs_hk_03_01–03_03 (Beurt

In [44]:
usage = response.usage()
print(f'Input Tokens: {usage.input_tokens} ({usage.input_tokens*1.75*1e-6:0.04f} $)')
print(f'Output Tokens: {usage.output_tokens} ({usage.output_tokens*14.0*1e-6:0.04f} $)')
print(f'Total Tokens: {usage.total_tokens} (total cost: {(usage.input_tokens*1.75*1e-6 + usage.input_tokens*1.75*1e-6):0.04f} $)')

Input Tokens: 33255 (0.0582 $)

Output Tokens: 605 (0.0085 $)

Total Tokens: 33860 (total cost: 0.1164 $)

In [12]:
settings = agent.model.settings
print(f'Settings: {settings}')
print(f'temperature: {settings.get('temperature')}')
print(f'presence_penalty: {settings.get('presence_penalty')}')
print(f'frequency_penalty: {settings.get('frequency_penalty')}')

Settings: {'temperature': 0.0, 'presence_penalty': 0.0, 'frequency_penalty': 0.0, 'parallel_tool_calls': True, 
'openai_reasoning_effort': 'none', 'openai_reasoning_summary': 'auto', 'openai_text_verbosity': 'low'}

temperature: 0.0

presence_penalty: 0.0

frequency_penalty: 0.0